In [8]:
!pip install findspark

In [9]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

In [10]:
#Libraries
import pandas as pd
from pyspark.sql.functions import row_number,lit ,desc, monotonically_increasing_id
from pyspark.sql.functions import desc, row_number, monotonically_increasing_id
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField,IntegerType, StringType , DateType,FloatType

In [11]:
# May take a little while on a local computer
spark = SparkSession.builder.appName("Basics").getOrCreate()

In [7]:
from google.colab import files
uploaded = files.upload()

Saving IcecreamDataset.xlsx to IcecreamDataset.xlsx


In [12]:
import pandas as pd
df = pd.read_excel("IcecreamDataset.xlsx")

In [13]:
df.head()

,SalesDate,SalesQty,SalesAmount,ProductCategory,ProductSubCategory,ProductName,StoreProvince,StoreRegion,StoreZone,StoreName,StoreArea,Route,PaymentTerms,SalesMan,Category
0,2022-09-02,2,36956.0,ICE CREAMS,MOMENT TUB,STRABERY CHEESE CAKE MOMENT,SINDH,KARACHI,ZONE-C,KOILA RESTURANT,C.P BARAR Town,ORANGI TOWN,Credit (114),ADBULLAH,RSO
1,2022-09-05,5,92390.0,ICE CREAMS,MOMENT TUB,CHOC FUDGE BROWNI MOMENT,SINDH,KARACHI,ZONE-F,S.M ANEES (AGHA SUPER),C.P BARAR Town,ORANGI TOWN,Credit (114),ADBULLAH,RSO
2,2022-01-24,25,40325.0,CONES,2 IN 1 CONE,KING CONE 2/1 (1X25),SINDH,KARACHI,LOWER SINDH,ALI MOHD.HOTEL,KPT Clinic,ORANGI TOWN,Credit (114),ADBULLAH,RSO
3,2022-01-15,16,25808.0,CUPS,CLASSIC CUP,MANGO CLASSIC CUP (1X16),SINDH,KARACHI,LOWER SINDH,NEW MURAD COLD,KPT Clinic,ORANGI TOWN,Credit (114),ADBULLAH,RSO
4,2022-02-11,12,13272.0,CUPS,CHEER CUP,CHEER VANILA CUP (1X24),SINDH,KARACHI,LOWER SINDH,ZEESHAN BAKERY,KPT Clinic,ORANGI TOWN,Credit (114),ADBULLAH,RSO


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46258 entries, 0 to 46257
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   SalesDate           46258 non-null  datetime64[ns]
 1   SalesQty            46258 non-null  int64         
 2   SalesAmount         46257 non-null  float64       
 3   ProductCategory     46258 non-null  object        
 4   ProductSubCategory  46258 non-null  object        
 5   ProductName         46258 non-null  object        
 6   StoreProvince       46258 non-null  object        
 7   StoreRegion         46257 non-null  object        
 8   StoreZone           46258 non-null  object        
 9   StoreName           46254 non-null  object        
 10  StoreArea           46191 non-null  object        
 11  Route               46241 non-null  object        
 12  PaymentTerms        46258 non-null  object        
 13  SalesMan            46258 non-null  object    

In [15]:
df.columns

Index(['SalesDate', 'SalesQty', 'SalesAmount', 'ProductCategory',
       'ProductSubCategory', 'ProductName', 'StoreProvince', 'StoreRegion',
       'StoreZone', 'StoreName', 'StoreArea', 'Route', 'PaymentTerms',
       'SalesMan', 'Category'],
      dtype='object')

In [16]:
#create schema for your dataframe
schema = StructType(
                   [StructField("SalesDate", DateType(), True)\
                   ,StructField("SalesQty",IntegerType(), True)\
                   ,StructField("SalesAmount", FloatType(), True)\
                   ,StructField("ProductCategory", StringType(), True)\
                   ,StructField("ProductSubCategory", StringType(), True)\
                   ,StructField("ProductName", StringType(), True)\
                   ,StructField("StoreName", StringType(), True)\
                   ,StructField("StoreRegion", StringType(), True)\
                   ,StructField("StoreProvince", StringType(), True)\
                   ,StructField("StoreZone", StringType(), True)\
                   ,StructField("StoreArea", StringType(), True)\
                   ,StructField("PaymentTerms", StringType(), True)\
                   ,StructField("SalesMan", StringType(), True)\
                   ,StructField("Route", StringType(), True)\
                   ,StructField("Category", StringType(), True)
                   ]
                   )

In [17]:
df2 = spark.createDataFrame(df,schema=schema)

In [18]:
df2.dtypes

[('SalesDate', 'date'),
 ('SalesQty', 'int'),
 ('SalesAmount', 'float'),
 ('ProductCategory', 'string'),
 ('ProductSubCategory', 'string'),
 ('ProductName', 'string'),
 ('StoreName', 'string'),
 ('StoreRegion', 'string'),
 ('StoreProvince', 'string'),
 ('StoreZone', 'string'),
 ('StoreArea', 'string'),
 ('PaymentTerms', 'string'),
 ('SalesMan', 'string'),
 ('Route', 'string'),
 ('Category', 'string')]

In [19]:
df2.createOrReplaceTempView('dim_product')
spark.sql('select ProductCategory,ProductSubCategory,ProductName from dim_product')

DataFrame[ProductCategory: string, ProductSubCategory: string, ProductName: string]

In [31]:
dimproduct = spark.sql('select ProductCategory,ProductSubCategory,ProductName from dim_product').distinct()

In [32]:
dimproduct.show()


+--------------------+------------------+--------------------+
|     ProductCategory|ProductSubCategory|         ProductName|
+--------------------+------------------+--------------------+
|               STICK|   STICK ICE LOLLY|      MAMAMIA (1X50)|
|          Blue Bell |         BALL CONE|    BALL CONE (1X20)|
|          Chapman's |        FUNDAY CUP|FUNDE CHOCLATE CU...|
| Casper's Ice Cream |           SUPREME|MANGO RIPPLE SUPREME|
|      Cows Creamery |       1 LITRE SPL|SHAHI KULFA PARTY...|
|            Breyers |   STICK ICE LOLLY|      MAMAMIA (1X50)|
|     Double Rainbow |    HALF LITRE ODR|STRAWBERRY FAMILY...|
|            Breyers |           2 LITRE|     TRIPPLE DELIGHT|
|          Blue Bell |       1 LITRE ODR|     ROSE PARTY PACK|
|          Creambell |        STICK LAVA|LAVA CHOCLATE (1X30)|
|            Braum's |           2 LITRE|     TRIPPLE DELIGHT|
|          D'Onofrio |        FUNDAY CUP|DIET STRABERRY CU...|
|Bonnie Doon Ice C...|   STICK ICE LOLLY|ICE CREAM SODA

In [33]:
dimproduct = dimproduct.withColumn('productID', row_number().over(Window.orderBy(monotonically_increasing_id())) )

In [34]:
dimproduct.show()

+--------------------+------------------+--------------------+---------+
|     ProductCategory|ProductSubCategory|         ProductName|productID|
+--------------------+------------------+--------------------+---------+
|               STICK|   STICK ICE LOLLY|      MAMAMIA (1X50)|        1|
|          Blue Bell |         BALL CONE|    BALL CONE (1X20)|        2|
|          Chapman's |        FUNDAY CUP|FUNDE CHOCLATE CU...|        3|
| Casper's Ice Cream |           SUPREME|MANGO RIPPLE SUPREME|        4|
|      Cows Creamery |       1 LITRE SPL|SHAHI KULFA PARTY...|        5|
|            Breyers |   STICK ICE LOLLY|      MAMAMIA (1X50)|        6|
|     Double Rainbow |    HALF LITRE ODR|STRAWBERRY FAMILY...|        7|
|            Breyers |           2 LITRE|     TRIPPLE DELIGHT|        8|
|          Blue Bell |       1 LITRE ODR|     ROSE PARTY PACK|        9|
|          Creambell |        STICK LAVA|LAVA CHOCLATE (1X30)|       10|
|            Braum's |           2 LITRE|     TRIPP

In [45]:
dim_Product = dimproduct.toPandas()
# 1. Save to Excel in the Colab temporary directory
dim_Product.to_excel("dim_product.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("dim_product.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
df2.createOrReplaceTempView("dim_store")
dimstore = spark.sql('select StoreName,StoreRegion,StoreProvince,StoreZone from dim_store').distinct()
dimstore = dimstore.withColumn('StoreID', row_number().over(Window.orderBy(monotonically_increasing_id())) )
dim_store = dimstore.toPandas()
dim_store = dim_store.reindex(columns = ['StoreID' , 'StoreName' ,'StoreProvince' ,'StoreRegion' , 'StoreZone' ])

In [37]:
# 1. Save to Excel in the Colab temporary directory
dim_store.to_excel("dim_store.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("dim_store.xlsx")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
df2.createOrReplaceTempView("dim_dealer")
dimdealer = spark.sql('select SalesMan,Route from dim_dealer').distinct()
dimdealer = dimdealer.withColumn('DealerID', row_number().over(Window.orderBy(monotonically_increasing_id())))
dim_dealer = dimdealer.toPandas()
dim_dealer = dim_dealer.reindex(columns=['DealerID', 'SalesMan' ,'Route'])

In [39]:
# 1. Save to Excel in the Colab temporary directory
dim_dealer.to_excel("dim_dealer.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("dim_dealer.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
def create_date(start='2010-01-01', end='2050-12-31'):
    df = pd.DataFrame({"date": pd.date_range(start, end)})
    df["day"] = df.date.dt.day
    df["month"] = df.date.dt.month
    df["week"] = df.date.dt.isocalendar().week
    df["quarter"] = df.date.dt.quarter
    df["year"] = df.date.dt.year
    df.insert(0, 'date_id', (df.year.astype(str) + df.month.astype(str).str.zfill(2) + df.day.astype(str).str.zfill(2)))
    return df

In [42]:
dim_date = create_date()
# 1. Save to Excel in the Colab temporary directory
dim_date.to_excel("dim_date.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("dim_date.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
df2.createOrReplaceTempView("lkp_payment")
lkp = spark.sql('select PaymentTerms from lkp_payment').distinct()
lkp = lkp.withColumn('PaymentID', row_number().over(Window.orderBy(monotonically_increasing_id())))
lkp_Payment = lkp.toPandas()

In [ ]:
dim_date = create_date()
# 1. Save to Excel in the Colab temporary directory
lkp_Payment.to_excel("lkp_Payment.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("lkp_Payment.xlsx")

In [46]:
dimp= spark.createDataFrame(dim_Product)
dimp.createOrReplaceTempView('dimp')

In [47]:
dimstore = spark.createDataFrame(dim_store)
dimstore.createOrReplaceTempView('dimst')
dimd = spark.createDataFrame(dim_dealer)
dimd.createOrReplaceTempView('dimd')
lkp= spark.createDataFrame(lkp_Payment)
lkp.createOrReplaceTempView('lkp')
dk = spark.createDataFrame(dim_date)
dk.createOrReplaceTempView('dk')

In [56]:
df2.createOrReplaceTempView('sales')

In [57]:
fact = spark.sql('select s.SalesQTY , s.SalesAmount , dp.productID , ds.StoreID , dd.DealerID ,lkp.PaymentID ,dk.date_id from sales s inner join dimp dp on s.ProductCategory = dp.ProductCategory and s.ProductSubCategory = dp.ProductSubCategory and s.ProductName = dp.ProductName inner join  dimst ds on s.StoreName = ds.StoreName and s.StoreRegion = ds.StoreRegion and s.StoreZone = ds.StoreZone inner join dimd dd on s.SalesMan = dd.SalesMan and s.Route = dd.Route inner join lkp on s.PaymentTerms = lkp.PaymentTerms inner join dk on  s.SalesDate = dk.date')

In [58]:
fact_sales= fact.toPandas()
fact_sales.to_excel("fact_sales.xlsx", header=True, index=False)

# 2. Download the file to your PC
from google.colab import files
files.download("fact_sales.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
fact.show()

+--------+-----------+---------+-------+--------+---------+--------+
|SalesQTY|SalesAmount|productID|StoreID|DealerID|PaymentID| date_id|
+--------+-----------+---------+-------+--------+---------+--------+
|     225|   311175.0|      114|   5393|      24|       18|20230210|
|     225|   311175.0|      114|   1251|      24|       18|20230210|
|       1|     6084.0|      119|   1882|      24|       18|20221001|
|       1|     6084.0|      119|    874|      24|       18|20221001|
|      32|    53920.0|      219|   1882|      24|       18|20221002|
|      32|    53920.0|      219|    874|      24|       18|20221002|
|      38|    82498.0|      288|   1220|      24|       18|20221107|
|      38|    82498.0|      288|    726|      24|       18|20221107|
|      38|    82498.0|      288|    122|      24|       18|20221107|
|      68|    75208.0|      354|   3220|      24|       18|20220906|
|      68|    75208.0|      354|   1498|      24|       18|20220906|
|      68|    75208.0|      354|  

In [61]:
spark.sql("SELECT COUNT(*) FROM sales").show()
spark.sql("SELECT COUNT(*) FROM dimp").show()
spark.sql("SELECT COUNT(*) FROM dimst").show()
spark.sql("SELECT COUNT(*) FROM dimd").show()
spark.sql("SELECT COUNT(*) FROM lkp").show()
spark.sql("SELECT COUNT(*) FROM dk").show()


+--------+
|count(1)|
+--------+
|   46258|
+--------+

+--------+
|count(1)|
+--------+
|     617|
+--------+

+--------+
|count(1)|
+--------+
|    6882|
+--------+

+--------+
|count(1)|
+--------+
|      35|
+--------+

+--------+
|count(1)|
+--------+
|      36|
+--------+

+--------+
|count(1)|
+--------+
|   14975|
+--------+



In [60]:
fact_sales.head()

,SalesQTY,SalesAmount,productID,StoreID,DealerID,PaymentID,date_id
0,225,311175.0,114,5393,24,18,20230210
1,225,311175.0,114,1251,24,18,20230210
2,1,6084.0,119,1882,24,18,20221001
3,1,6084.0,119,874,24,18,20221001
4,32,53920.0,219,1882,24,18,20221002


In [62]:
spark.sql("SELECT DISTINCT ProductName FROM sales").show()
spark.sql("SELECT DISTINCT ProductName FROM dimp").show()


+--------------------+
|         ProductName|
+--------------------+
|COOKIES N CREAM M...|
|        DIPSY (1X60)|
|DIET STRABERRY CU...|
|PINE APPLE BULK PACK|
|   CHOC CHIP SUPREME|
|    BALL CONE (1X20)|
|LAVA CHOCLATE (1X30)|
|CHEER ROSE CUP (1...|
|  BLUE BERRY SUPREME|
|CARAMAL CRUNCH BU...|
|  MOMENT CONE (1X25)|
|SHAHI KULFA FAMIL...|
|VANILLA FUDGE SUP...|
|ICE CREAM SODA (1...|
|WHITE VANILLA BUL...|
|PINE APPLE BULK P...|
|SHAHI KULFA SOFT ...|
|     ROSE PARTY PACK|
| VANILLA FAMILY PACK|
|   KARAMAL BULK PACK|
+--------------------+
only showing top 20 rows

+--------------------+
|         ProductName|
+--------------------+
|COOKIES N CREAM M...|
|        DIPSY (1X60)|
|DIET STRABERRY CU...|
|PINE APPLE BULK PACK|
|   CHOC CHIP SUPREME|
|    BALL CONE (1X20)|
|LAVA CHOCLATE (1X30)|
|CHEER ROSE CUP (1...|
|  BLUE BERRY SUPREME|
|CARAMAL CRUNCH BU...|
|  MOMENT CONE (1X25)|
|SHAHI KULFA FAMIL...|
|ICE CREAM SODA (1...|
|VANILLA FUDGE SUP...|
|WHITE VANILLA BUL...|
|PINE AP

In [63]:
print(df2.show())

+----------+--------+-----------+--------------------+------------------+--------------------+---------+-----------+-------------+--------------------+--------------------+------------+------------+--------+--------+
| SalesDate|SalesQty|SalesAmount|     ProductCategory|ProductSubCategory|         ProductName|StoreName|StoreRegion|StoreProvince|           StoreZone|           StoreArea|PaymentTerms|    SalesMan|   Route|Category|
+----------+--------+-----------+--------------------+------------------+--------------------+---------+-----------+-------------+--------------------+--------------------+------------+------------+--------+--------+
|2022-09-02|       2|    36956.0|          ICE CREAMS|        MOMENT TUB|STRABERY CHEESE C...|    SINDH|    KARACHI|       ZONE-C|     KOILA RESTURANT|      C.P BARAR Town| ORANGI TOWN|Credit (114)|ADBULLAH|     RSO|
|2022-09-05|       5|    92390.0|          ICE CREAMS|        MOMENT TUB|CHOC FUDGE BROWNI...|    SINDH|    KARACHI|       ZONE-F|S.

In [66]:
dim_Product.head()

,ProductCategory,ProductSubCategory,ProductName,productID
0,STICK,STICK ICE LOLLY,MAMAMIA (1X50),1
1,Blue Bell,BALL CONE,BALL CONE (1X20),2
2,Chapman's,FUNDAY CUP,FUNDE CHOCLATE CUP (1X16),3
3,Casper's Ice Cream,SUPREME,MANGO RIPPLE SUPREME,4
4,Cows Creamery,1 LITRE SPL,SHAHI KULFA PARTY PACK,5


In [67]:
dim_date['year']

,year
0,2010
1,2010
2,2010
3,2010
4,2010
...,...
14970,2050
14971,2050
14972,2050
14973,2050


In [68]:
dim_dealer.head()

,DealerID,SalesMan,Route
0,1,Credit (114),AMAN AFZAL
1,2,Cash,AIJAZ AHMED
2,3,With in 30 Days(110),GHULAM ABBAS
3,4,Cash,AMAN AFZAL
4,5,With in 30 Days(110),FEROZ KHAN


In [70]:
dim_store.tail()

,StoreID,StoreName,StoreProvince,StoreRegion,StoreZone
6877,6878,SINDH,ZONE-F,KARACHI,PANAWALA MEDICAL
6878,6879,SINDH,ZONE-B,KARACHI,AL MADINA
6879,6880,SINDH,LOWER SINDH,HYDERABAD,HYDER BOOK STORE
6880,6881,SINDH,UPPER SINDH,HYDERABAD,MAKKA ICE CREAM SPOT
6881,6882,SINDH,UPPER SINDH,HYDERABAD,FLAMINGO JUICE
